In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, stddev, min, max, lit, count, to_date, window
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType

# Initialize Spark session
spark = SparkSession.builder.appName("WindTurbinePipeline").getOrCreate()

# Paths for Delta Lake layers
BRONZE_PATH = "/delta/bronze/tbl_wind_turbines"
SILVER_PATH = "/delta/silver/tbl_wind_turbines"
GOLD_PATH = "/delta/gold/daily_summary_statistics"
ANOMALY_PATH = "/delta/silver/anomalies/tbl_wind_turbines"

# Schema definition for raw data
schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("turbine_id", IntegerType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("wind_direction", IntegerType(), True),
    StructField("power_output", DoubleType(), True)
])

In [0]:
# data_path = "s3://databricks-workspace-stack-fa72e-bucket/unity-catalog/2498248527665908/data_group_1.csv"
def ingest_bronze_layer(data_path):
    """Ingest raw data into the Bronze layer."""
    # Read the raw data into DF using the defined schema
    raw_df = spark.read.csv(data_path, header=True, schema=schema)
    #Store the raw data into Bronze table
    raw_df.write.format("delta").mode("overwrite").save(BRONZE_PATH)
    print("Bronze layer data ingested.")
    # raw_df.display()

In [0]:
def process_silver_layer():
    """Cleanse and transform data in the Silver layer."""
    bronze_df = spark.read.format("delta").load(BRONZE_PATH)

    # Handle missing values by imputing with mean or mode as a replacement(could be different depending on agreed logic from business)
    for column in ["wind_speed", "wind_direction", "power_output"]:
        if column in ["wind_speed", "power_output"]:
            mean_value = bronze_df.select(mean(col(column)).alias("mean_value")).collect()[0]["mean_value"]
            bronze_df = bronze_df.fillna({column: mean_value})
        elif column == "wind_direction":
            mode_value = bronze_df.groupBy(col(column)).count().orderBy(col("count").desc()).first()[column]
            bronze_df = bronze_df.fillna({column: mode_value})

    # Remove invalid values (e.g., negative wind speed or power output). also this is subject to agreement from business either to completely remove or moove to anomaly table
    bronze_df = bronze_df.filter((col("wind_speed") >= 0) & (col("power_output") >= 0))

    # Group by turbine_id and calculate mean and standard deviation for each turbine
    stats_df = bronze_df.groupBy("turbine_id").agg(
        mean("power_output").alias("mean_power"),
        stddev("power_output").alias("std_dev_power")
    )

    # Join the statistics back to the original DataFrame
    bronze_df = bronze_df.join(stats_df, on="turbine_id")

    # Add a column to flag anomalies based on the 2-standard-deviation rule
    bronze_df = bronze_df.withColumn(
        "is_anomaly",
        (col("power_output") > col("mean_power") + 2 * col("std_dev_power")) |
        (col("power_output") < col("mean_power") - 2 * col("std_dev_power"))    )

    #Filter for anomaly rows to be ingested into anomaly table
    df_with_anomalies = bronze_df.filter(col("is_anomaly") == True).drop("is_anomaly")

    # Store the anomalies in the Silver layer or a separate anomalies table
    df_with_anomalies.write.format("delta").mode("overwrite").save(ANOMALY_PATH)

    #Filter rows with no anomalies for Silver ingestion and drop calculated columns
    silver_df = bronze_df.filter(col("is_anomaly") == False).drop("is_anomaly").drop("mean").drop("stddev")
    silver_df.write.format("delta").mode("overwrite").save(SILVER_PATH)
    print("Silver layer data processed.")


In [0]:
def process_gold_layer():
    """Generate summary statistics in the Gold layer."""
    silver_df = spark.read.format("delta").load(SILVER_PATH)

    # Calculate daily summary statistics
    daily_summary_df = silver_df.withColumn("date", to_date(col("timestamp"))).groupBy("date", "turbine_id").agg(
        min("power_output").alias("min_power"),
        max("power_output").alias("max_power"),
        mean("power_output").alias("avg_power")
    )
       
    # Save to Gold layer
    daily_summary_df.write.format("delta").mode("overwrite").save(GOLD_PATH)

    print("Gold layer data processed.")


In [0]:
%python
def main():
    #Path to raw data
    data_path = "/data/raw/"  

    # Step 1: Ingest raw data into Bronze layer
    ingest_bronze_layer(data_path)

    # Step 2: Process data into Silver layer and detect anomalies in Silver layer
    process_silver_layer()

    # Step 3: Generate summary statistics 
    process_gold_layer()

if __name__ == "__main__":
    main()


Bronze layer data ingested.
Silver layer data processed.
Gold layer data processed.


In [0]:
# Testing the data in the tables
# delta_table_path = "/delta/silver/tbl_wind_turbines/"
# delta_table_path = "/delta/bronze/tbl_wind_turbines/"
# delta_table_path = "/delta/gold/daily_summary_statistics/"
delta_table_path = "/delta/silver/anomalies/tbl_wind_turbines"

# Load Delta table into a DataFrame
df = spark.read.format("delta").load(delta_table_path)

# df = df.filter(col("power_output") >= 30)

# Show DataFrame content
df.show()

+----------+-------------------+----------+--------------+------------+------------------+-----------------+
|turbine_id|          timestamp|wind_speed|wind_direction|power_output|        mean_power|    std_dev_power|
+----------+-------------------+----------+--------------+------------+------------------+-----------------+
|         5|2022-03-31 23:00:00|      13.5|            55|       300.0|3.4104302477183834|10.75787746822921|
|         4|2022-03-01 01:00:00|      13.9|           170|        60.0|3.0223958333333307|2.240650513101064|
+----------+-------------------+----------+--------------+------------+------------------+-----------------+



In [0]:
# List all Delta tables (assuming they are stored under a directory like "/mnt/delta/")
tables_directory = "/delta/"
tables = dbutils.fs.ls(tables_directory)

# Loop through all Delta tables and remove them safely
for table in tables:
    if table.isDir():
        table_name = table.name.strip("/")  # Remove leading/trailing slashes
        try:
            # Drop the Delta table from the catalog (if it's registered as a table)
            spark.sql(f"DROP TABLE IF EXISTS {table_name}")
            print(f"Table {table_name} dropped successfully.")
            
            # Remove the directory and its transaction logs
            dbutils.fs.rm(table.path, recurse=True)
            print(f"Delta table directory {table.path} removed successfully.")
        except Exception as e:
            print(f"Error while removing table {table_name}: {e}")